
# Bagaev 2019 reduced reproduction notebook

This notebook is a **fully runnable reduced reproduction** of the key claim from Bagaev et al. (2019):

> higher basal nuclear NF-kB in primary macrophages accelerates the time-to-peak of NF-kB nuclear translocation after LPS stimulation.

## What this notebook reproduces

This notebook reproduces a **Figure 3D-style panel** using a compact ODE model that is calibrated to the paper's textual anchors for low-LPS stimulation.

It is **not** a full SBML replay of the BioModels submission. The exact SBML model is available in BioModels as `MODEL1809230001`, but the public XML file is not bundled here. This notebook is therefore intended as a **traceable first reproducibility milestone** for the library.

## Literature anchors used here

The paper reports that, at low LPS, increasing basal nuclear NF-kB shortens the time to peak. In the article text, the reported peak times are approximately:

- 0% basal NF-kB -> 110 min
- 10% basal NF-kB -> 75 min
- 30% basal NF-kB -> 55 min

These values are used below as calibration targets for the reduced model.


In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

plt.rcParams['figure.dpi'] = 140
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False



## Reduced model

We use a one-variable ODE for nuclear NF-kB, `N(t)`, driven by a transient activation kernel `A(t)`:

\[
rac{dN}{dt} = k_{imp}(1 + lpha N_0)A(t)(1-N) - k_{exp}N
\]

with

\[
A(t) = \left(1-e^{-t/	au_{on}}ight)e^{-t/	au_{off}}
\]

Here:

- `N(t)` is nuclear NF-kB as a fraction of total NF-kB
- `N0` is the basal nuclear NF-kB fraction before stimulation
- `k_imp` is an effective import term
- `k_exp` is an effective export/relaxation term
- `alpha` controls how much pre-activation accelerates translocation

This model is intentionally minimal. Its purpose is to reproduce the **qualitative and semi-quantitative timing effect** emphasized in Bagaev et al.


In [ ]:

# Calibrated parameters for the reduced reproduction
params = {
    'k_imp': 0.04,
    'k_exp': 0.0025,
    'tau_on': 12.0,
    'tau_off': 55.0,
    'alpha': 16.0,
}

time = np.linspace(0, 180, 1801)

def activation_kernel(t, tau_on, tau_off):
    return (1.0 - np.exp(-t / tau_on)) * np.exp(-t / tau_off)

def simulate_nfkb(basal_fraction, time_grid=time, p=params):
    def rhs(t, y):
        N = y[0]
        A = activation_kernel(t, p['tau_on'], p['tau_off'])
        dNdt = p['k_imp'] * (1.0 + p['alpha'] * basal_fraction) * A * (1.0 - N) - p['k_exp'] * N
        return [dNdt]

    sol = solve_ivp(rhs, (time_grid.min(), time_grid.max()), [basal_fraction], t_eval=time_grid, method='LSODA')
    return pd.DataFrame({'time_min': sol.t, 'nuclear_nfkb_fraction': sol.y[0]})

def peak_time_minutes(df):
    idx = df['nuclear_nfkb_fraction'].idxmax()
    return float(df.loc[idx, 'time_min'])


## Check the calibration targets from the paper text

In [ ]:

text_targets = pd.DataFrame({
    'basal_fraction': [0.00, 0.10, 0.30],
    'reported_time_to_peak_min': [110.0, 75.0, 55.0],
    'source_note': [
        'Paper text: low LPS, 0% basal -> 110 min',
        'Paper text: low LPS, 10% basal -> 75 min',
        'Paper text: low LPS, 30% basal -> 55 min',
    ]
})

predicted = []
for basal in text_targets['basal_fraction']:
    df = simulate_nfkb(basal)
    predicted.append(peak_time_minutes(df))

comparison = text_targets.copy()
comparison['reduced_model_peak_min'] = predicted
comparison['absolute_error_min'] = (comparison['reduced_model_peak_min'] - comparison['reported_time_to_peak_min']).abs()
comparison


## Reproduce the low-LPS basal NF-kB timing effect (Figure 3D-style)

In [ ]:

basal_levels = [0.00, 0.10, 0.20, 0.30]
curves = {b: simulate_nfkb(b) for b in basal_levels}

fig, ax = plt.subplots(figsize=(7, 4.5))
for basal, df in curves.items():
    ax.plot(df['time_min'], df['nuclear_nfkb_fraction'], label=f'Basal = {int(basal*100)}%')

ax.set_xlabel('Time after LPS stimulation (min)')
ax.set_ylabel('Nuclear NF-kB (fraction of total)')
ax.set_title('Reduced reproduction of Bagaev 2019 basal NF-kB timing effect')
ax.legend(frameon=False)
plt.show()


## Report model-derived peak times

In [ ]:

peak_summary = pd.DataFrame({
    'basal_fraction': basal_levels,
    'time_to_peak_min': [peak_time_minutes(curves[b]) for b in basal_levels],
    'peak_nuclear_nfkb_fraction': [curves[b]['nuclear_nfkb_fraction'].max() for b in basal_levels],
})
peak_summary



## Save outputs for the repository

This cell writes two CSV files that you can place in your model folder:

- `data/processed/reduced_reproduction_targets.csv`
- `data/processed/reduced_reproduction_peak_summary.csv`


In [ ]:

from pathlib import Path

outdir = Path('bagaev2019_outputs')
outdir.mkdir(exist_ok=True)

comparison.to_csv(outdir / 'reduced_reproduction_targets.csv', index=False)
peak_summary.to_csv(outdir / 'reduced_reproduction_peak_summary.csv', index=False)

print('Saved files:')
for p in sorted(outdir.glob('*.csv')):
    print('-', p)



## Interpretation

This reduced model reproduces the central qualitative result from Bagaev et al.: **higher basal nuclear NF-kB produces earlier NF-kB translocation peaks under weak stimulation**.

### What this notebook does well

It gives you a working, transparent, and traceable first notebook for the library. It also creates machine-readable output files that can live beside your curated model.

### What it does not yet do

It does not reconstruct the full published SBML model, and it does not yet replay the authors' exact fitted experimental curves from Figure 3B. To do that, the next upgrade should add:

1. the BioModels SBML XML file (`MODEL1809230001`),
2. exact digitized experimental points or original tabular data, and
3. an SBML-capable simulator such as libSBML + roadrunner/tellurium.
